In [2]:
from google.colab import files
uploaded = files.upload()

Saving Test.csv to Test.csv
Saving Train.csv to Train.csv


Step 2

In [4]:
import pandas as pd

# Load the Train.csv file into a pandas DataFrame
try:
    df_train = pd.read_csv('Train.csv')
    print('Train.csv loaded successfully. Here is its structure:')
    print(df_train.info())
    print('\nFirst 5 rows of Train.csv:')
    print(df_train.head())
except FileNotFoundError:
    print("Error: Train.csv not found. Please ensure it's uploaded correctly.")
except Exception as e:
    print(f"An error occurred while reading Train.csv: {e}")

Train.csv loaded successfully. Here is its structure:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10001 entries, 0 to 10000
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   tweet_id   10001 non-null  object 
 1   safe_text  10001 non-null  object 
 2   label      10000 non-null  float64
 3   agreement  9999 non-null   float64
dtypes: float64(2), object(2)
memory usage: 312.7+ KB
None

First 5 rows of Train.csv:
   tweet_id                                          safe_text  label  \
0  CL1KWCMY  Me &amp; The Big Homie meanboy3000 #MEANBOY #M...    0.0   
1  E3303EME  I'm 100% thinking of devoting my career to pro...    1.0   
2  M4IVFSMS  #whatcausesautism VACCINES, DO NOT VACCINATE Y...   -1.0   
3  1DR6ROZ4  I mean if they immunize my kid with something ...   -1.0   
4  J77ENIIE  Thanks to <user> Catch me performing at La Nui...    0.0   

   agreement  
0        1.0  
1        1.0  
2        1.0  
3        1.0

Step 4

In [6]:
import pandas as pd

# Load datasets
train = pd.read_csv("Train.csv")
test = pd.read_csv("Test.csv")

# View first few rows
print(train.head())

# Check label distribution (assuming 'label' is the sentiment column)
print(train['label'].value_counts())

   tweet_id                                          safe_text  label  \
0  CL1KWCMY  Me &amp; The Big Homie meanboy3000 #MEANBOY #M...    0.0   
1  E3303EME  I'm 100% thinking of devoting my career to pro...    1.0   
2  M4IVFSMS  #whatcausesautism VACCINES, DO NOT VACCINATE Y...   -1.0   
3  1DR6ROZ4  I mean if they immunize my kid with something ...   -1.0   
4  J77ENIIE  Thanks to <user> Catch me performing at La Nui...    0.0   

   agreement  
0        1.0  
1        1.0  
2        1.0  
3        1.0  
4        1.0  
label
 0.000000    4908
 1.000000    4053
-1.000000    1038
 0.666667       1
Name: count, dtype: int64


Step 5: Clean and Prepare the Data

In [10]:

import re
import nltk
from nltk.corpus import stopwords

# Download stopwords if not already downloaded
try:
    stop_words = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords')
    stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Ensure text is a string to handle potential float/NaN values
    text = str(text)
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # Remove punctuation and numbers
    text = ' '.join(word for word in text.split() if word not in stop_words)
    return text

# Fill NaN values in 'safe_text' with empty strings before applying clean_text
train['clean_text'] = train['safe_text'].fillna('').apply(clean_text)
test['clean_text'] = test['safe_text'].fillna('').apply(clean_text)

print(train[['safe_text', 'clean_text']].head())

                                           safe_text  \
0  Me &amp; The Big Homie meanboy3000 #MEANBOY #M...   
1  I'm 100% thinking of devoting my career to pro...   
2  #whatcausesautism VACCINES, DO NOT VACCINATE Y...   
3  I mean if they immunize my kid with something ...   
4  Thanks to <user> Catch me performing at La Nui...   

                                          clean_text  
0  amp big homie meanboy meanboy mb mbs mmr stegm...  
1  im thinking devoting career proving autism isn...  
2          whatcausesautism vaccines vaccinate child  
3  mean immunize kid something wont secretly kill...  
4  thanks user catch performing la nuit nyc st av...  


Step 6: Convert Text to Numerical Data

In [13]:
# Machine learning models cannot work directly with text.
# Use TF-IDF Vectorization to transform text into numerical features.

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train['clean_text'])
X_test = vectorizer.transform(test['clean_text'])

y_train = train['label']

Step 7: Train a Classification Model

In [17]:
# You can start with Logistic Regression, a simple and effective baseline model.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Identify non-NaN indices in y_train
non_nan_indices = y_train.dropna().index

# Filter X_train and y_train to remove rows with NaN labels
X_train_filtered = X_train[non_nan_indices]
y_train_filtered = y_train.loc[non_nan_indices]

# Convert labels to integer type to ensure they are treated as discrete classes
y_train_filtered = y_train_filtered.astype(int)

# Train model
model = LogisticRegression(max_iter=200)
model.fit(X_train_filtered, y_train_filtered)

# Predict on training data (using filtered X_train and y_train)
y_pred = model.predict(X_train_filtered)
print(classification_report(y_train_filtered, y_pred))

              precision    recall  f1-score   support

          -1       0.88      0.38      0.53      1038
           0       0.85      0.89      0.87      4909
           1       0.79      0.87      0.82      4053

    accuracy                           0.82     10000
   macro avg       0.84      0.71      0.74     10000
weighted avg       0.83      0.82      0.82     10000



Step 8: Create Final Predictions

In [20]:
# Generate predictions for the test dataset.

# Predict on test set
test_predictions = model.predict(X_test)

# Prepare submission file
submission = pd.DataFrame({
    'tweet_id': test['tweet_id'],
    'sentiment': test_predictions
})

submission.to_csv("submission.csv", index=False)
print("Submission file created successfully!")

Submission file created successfully!
